# Sprint 10 · Webinar 28 · Data Analytics teórico  
## Simulación como herramienta estadística

**Programa:** Data Analytics  
**Sprint:** 10  
**Duración:** 100 minutos  
**Modalidad:** Teórica (con demostraciones en Python)


## Fecha
**Por definir**


## Objetivos de la sesión

Al finalizar la sesión, el/la estudiante podrá:

1. Explicar **qué es una simulación** y por qué es una herramienta poderosa en análisis de datos.
2. Generar **datos sintéticos** que respeten una estructura estadística conocida.
3. Aplicar **remuestreo (bootstrap)** sobre datos reales para estimar incertidumbre sin fórmulas complicadas.
4. Usar **simulación de Monte Carlo** para responder preguntas de negocio.
5. Emplear simulación para **chequear supuestos** de pruebas estadísticas (test de permutaciones).
6. Construir un **mini-proyecto integrador** que combine todas las técnicas anteriores.


## Agenda sugerida (100 minutos)

| Tiempo | Bloque | Contenido |
|---:|---|---|
| 0–10 | Actividad 0 | ¿Qué es simular? Intuición con un dado justo vs. cargado |
| 10–25 | Ejercicio 1 | Generar datos sintéticos: crear un dataset realista desde cero |
| 25–45 | Ejercicio 2 | Bootstrap: remuestrear datos reales para estimar intervalos de confianza |
| 45–65 | Ejercicio 3 | Simulación de Monte Carlo: estimar probabilidades de negocio |
| 65–85 | Ejercicio 4 | Test de permutaciones: chequear hipótesis sin supuestos paramétricos |
| 85–100 | Proyecto integrador | Unir todo: ¿el nuevo modelo de pricing es mejor? |


---
## Actividad 0 · Calentamiento: ¿Qué es simular? (10 min)

### Introducción teórica

**Simular** es generar datos artificiales usando reglas conocidas (distribuciones, probabilidades, modelos) para:
- **Explorar** escenarios que aún no han ocurrido.
- **Estimar** cantidades difíciles de calcular con fórmulas.
- **Validar** si nuestras pruebas estadísticas funcionan correctamente.

> **Analogía:** imagina que quieres saber si un dado es justo. Podrías:
> 1. Tirarlo 10,000 veces y ver si cada cara sale ~1/6 del tiempo (simulación física).
> 2. O pedirle a Python que "tire" el dado 10,000 veces (simulación computacional).

La ventaja de simular en la computadora: es **rápido**, **reproducible** y puedes cambiar las reglas fácilmente.

### Tipos de simulación que veremos hoy

| Técnica | ¿Qué hace? | ¿Para qué sirve? |
|---|---|---|
| Datos sintéticos | Genera datos falsos con estructura conocida | Probar pipelines, practicar, prototipar |
| Bootstrap (remuestreo) | Saca muchas muestras **del mismo dataset** con reemplazo | Intervalos de confianza sin fórmulas |
| Monte Carlo | Repite un proceso aleatorio miles de veces | Estimar probabilidades, riesgo financiero |
| Test de permutaciones | Mezcla las etiquetas de dos grupos al azar | Probar hipótesis sin supuestos paramétricos |

### Actividad en grupo (5 min)

En equipos de 2-3, discutan:

1. ¿En qué situación de negocio les gustaría "simular" antes de tomar una decisión?
2. ¿Por qué no basta con mirar los datos que ya tenemos?

**Recurso externo:** para visualizar distribuciones de probabilidad de forma interactiva, explora:  
[GeoGebra – Estadística](https://www.geogebra.org/math/statistics?lang=en)  
Puedes jugar con dados, monedas y distribuciones normales para construir intuición.


In [ ]:
# ============================================================
# Imports y configuración (ejecuta esta celda primero)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# Semilla global para reproducibilidad
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Librerías cargadas correctamente.")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

In [ ]:
# ============================================================
# Ejemplo rápido: simular un dado justo vs. un dado cargado
# ============================================================

n_lanzamientos = 10_000

# Dado justo: cada cara tiene probabilidad 1/6
dado_justo = rng.integers(1, 7, size=n_lanzamientos)  # 7 no se incluye

# Dado cargado: el 6 sale el doble de veces
# Probabilidades: 1/7 para caras 1-5, 2/7 para cara 6
probs_cargado = [1/7, 1/7, 1/7, 1/7, 1/7, 2/7]
dado_cargado = rng.choice([1, 2, 3, 4, 5, 6], size=n_lanzamientos, p=probs_cargado)

# Visualizar resultados
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, datos, titulo in zip(axes, 
                              [dado_justo, dado_cargado], 
                              ["Dado justo", "Dado cargado (6 favorecido)"]):
    # Contar frecuencia de cada cara
    caras, conteos = np.unique(datos, return_counts=True)
    frecuencias = conteos / n_lanzamientos
    
    ax.bar(caras, frecuencias, color="steelblue", edgecolor="white")
    ax.axhline(y=1/6, color="red", linestyle="--", label="Esperado (justo)")
    ax.set_xlabel("Cara del dado")
    ax.set_ylabel("Frecuencia relativa")
    ax.set_title(titulo)
    ax.set_xticks(range(1, 7))
    ax.legend()

plt.tight_layout()
plt.show()

print(f"Promedio dado justo:   {dado_justo.mean():.3f}  (esperado: 3.500)")
print(f"Promedio dado cargado: {dado_cargado.mean():.3f}  (esperado: ~3.714)")

> **Reflexión:** con solo 10,000 lanzamientos simulados pudimos detectar que el dado cargado se comporta diferente.  
> Esta es la idea central de la sesión: **usar la computadora para generar muchos datos y aprender de ellos.**


---
## Ejercicio 1 · Generar datos sintéticos realistas (15 min)

### Introducción teórica

Los **datos sintéticos** son datos generados por computadora que imitan la estructura y las propiedades estadísticas de datos reales, pero sin contener información real de ninguna persona o empresa.

#### ¿Para qué sirven?
- **Practicar** técnicas de análisis cuando no tienes datos reales disponibles.
- **Prototipar** dashboards y modelos antes de que lleguen los datos reales.
- **Proteger privacidad:** compartir datos con estructura similar sin exponer información sensible.
- **Controlar la verdad:** como TÚ generas los datos, sabes exactamente qué efecto existe — perfecto para aprender.

#### Distribuciones que usaremos

| Distribución | ¿Cuándo usarla? | Ejemplo |
|---|---|---|
| Normal | Variables continuas simétricas | Alturas, pesos, errores de medición |
| Binomial/Bernoulli | Resultados sí/no | Convertir o no, click o no click |
| Exponencial | Tiempos de espera | Tiempo entre compras |
| Uniforme | Cuando todo es igual de probable | IDs, fechas aleatorias |
| Poisson | Conteos por período | Visitas por hora, tickets por día |

### Objetivo
Generar un dataset de ventas de una tienda online con estructura controlada que usaremos en el resto de la clase.

### Pistas
- Usa `np.random.default_rng(seed)` para que los resultados sean reproducibles.
- Piensa en qué distribución es más realista para cada variable.
- Introduce un efecto conocido (ej.: un canal de marketing que vende más) para poder validarlo después.


In [ ]:
# ============================================================
# Ejercicio 1: Generar dataset sintético de ventas online
# ============================================================
# Contexto de negocio:
#   Una tienda online quiere analizar si su nuevo modelo de pricing
#   ("premium") genera más ingresos que el modelo actual ("estándar").
#   Generamos datos que simulan 6 meses de ventas.
# ============================================================

def generar_dataset_ventas(n_rows=8000, seed=42):
    """
    Genera un dataset sintético de ventas de e-commerce.
    
    Efectos controlados (para validar después):
    - El modelo 'premium' tiene un ticket promedio ~15% mayor.
    - El canal 'email' tiene mejor tasa de conversión.
    - Los fines de semana tienen más ventas impulsivas (mayor variabilidad).
    """
    rng = np.random.default_rng(seed)
    
    # --- Variables categóricas ---
    # Modelo de pricing: 50/50 estándar vs premium
    modelo_pricing = rng.choice(["estandar", "premium"], size=n_rows)
    
    # Canal de adquisición (distribución realista)
    canales = ["organic", "paid_search", "email", "social", "referral"]
    prob_canales = [0.30, 0.25, 0.20, 0.15, 0.10]
    canal = rng.choice(canales, size=n_rows, p=prob_canales)
    
    # Dispositivo
    dispositivo = rng.choice(["mobile", "desktop", "tablet"], 
                             size=n_rows, p=[0.55, 0.35, 0.10])
    
    # --- Fechas (6 meses) ---
    fecha_inicio = pd.Timestamp("2025-07-01")
    dias_offset = rng.integers(0, 180, size=n_rows)
    fechas = fecha_inicio + pd.to_timedelta(dias_offset, unit="D")
    dia_semana = fechas.dayofweek  # 0=lunes, 6=domingo
    es_finde = (dia_semana >= 5).astype(int)
    
    # --- Variable numérica: monto de venta (ticket) ---
    # Efecto del pricing: premium tiene media más alta
    ticket_base = np.where(
        modelo_pricing == "premium",
        rng.normal(loc=75, scale=25, size=n_rows),   # premium: media 75
        rng.normal(loc=65, scale=20, size=n_rows)    # estándar: media 65
    )
    
    # Efecto fin de semana: más variabilidad (compras impulsivas)
    ruido_finde = np.where(es_finde == 1, rng.normal(0, 10, n_rows), 0)
    ticket = np.maximum(ticket_base + ruido_finde, 5)  # mínimo $5
    ticket = np.round(ticket, 2)
    
    # --- Variable binaria: conversión ---
    # Probabilidad base + efecto canal email + efecto premium
    prob_conversion = 0.12  # tasa base 12%
    prob_conv = np.full(n_rows, prob_conversion)
    prob_conv = np.where(canal == "email", prob_conv + 0.08, prob_conv)    # email +8pp
    prob_conv = np.where(modelo_pricing == "premium", prob_conv + 0.03, prob_conv)  # premium +3pp
    convertido = rng.binomial(1, prob_conv)
    
    # --- Variable de conteo: páginas vistas (Poisson) ---
    paginas_vistas = rng.poisson(lam=5, size=n_rows)  # media ~5 páginas
    
    # --- Tiempo en sitio: exponencial ---
    tiempo_minutos = rng.exponential(scale=4.5, size=n_rows)  # media ~4.5 min
    tiempo_minutos = np.round(tiempo_minutos, 1)
    
    # --- Introducir missing values controlados (~3%) ---
    ticket_con_na = ticket.copy().astype(float)
    mask_na = rng.random(n_rows) < 0.03
    ticket_con_na[mask_na] = np.nan
    
    # --- Ensamblar DataFrame ---
    df = pd.DataFrame({
        "fecha": fechas,
        "modelo_pricing": modelo_pricing,
        "canal": canal,
        "dispositivo": dispositivo,
        "es_finde": es_finde,
        "paginas_vistas": paginas_vistas,
        "tiempo_minutos": tiempo_minutos,
        "ticket_usd": ticket_con_na,
        "convertido": convertido
    })
    
    return df

# Generar el dataset
df = generar_dataset_ventas(n_rows=8000, seed=RANDOM_SEED)
print(f"Dataset generado: {df.shape[0]} filas × {df.shape[1]} columnas")

In [ ]:
# Exploración rápida del dataset

print("Primeras filas:")
display(df.head())

In [ ]:
print("\nTipos de variables:")
display(df.dtypes)

In [ ]:
print("\nEstadísticas descriptivas:")
display(df.describe())

In [ ]:
print("\nPorcentaje de missing values por columna:")
display((df.isna().mean().sort_values(ascending=False) * 100).to_frame("missing_%").round(2))

In [ ]:
# Verificar los efectos que "escondimos" en la generación

print("=== Ticket promedio por modelo de pricing ===")
print(df.groupby("modelo_pricing")["ticket_usd"].mean().round(2))
print()

print("=== Tasa de conversión por canal ===")
print(df.groupby("canal")["convertido"].mean().round(4))
print()

print("=== Tasa de conversión por modelo ===")
print(df.groupby("modelo_pricing")["convertido"].mean().round(4))

> **Punto clave:** como NOSOTROS generamos los datos, sabemos que el efecto premium existe (media ~75 vs ~65 USD).  
> Esto es una ventaja enorme para aprender: podemos verificar si nuestras herramientas lo detectan correctamente.

> **Recurso:** Para explorar cómo se ven las distribuciones normal, exponencial y Poisson, visita:  
> [Seeing Theory – Distribuciones de probabilidad (Brown University)](https://seeing-theory.brown.edu/probability-distributions/index.html)  
> Es una herramienta visual e interactiva muy útil para construir intuición.


---
## Ejercicio 2 · Bootstrap: remuestrear para medir incertidumbre (20 min)

### Introducción teórica

El **bootstrap** es una técnica de remuestreo que nos permite estimar la **incertidumbre** de cualquier estadístico (media, mediana, diferencia de medias, etc.) **sin necesidad de fórmulas matemáticas complejas**.

#### ¿Cómo funciona?

1. Tomas tu muestra original de tamaño *n*.
2. Sacas una nueva muestra de tamaño *n* **con reemplazo** (algunos datos se repiten, otros quedan fuera).
3. Calculas el estadístico que te interesa (ej.: la media).
4. Repites los pasos 2-3 muchas veces (ej.: 5,000 veces).
5. Los resultados forman una **distribución bootstrap** → de ahí sacas intervalos de confianza.

#### ¿Por qué es tan útil?

- **No necesitas suponer normalidad** (ni ninguna distribución particular).
- Funciona para **cualquier estadístico** (media, mediana, percentiles, ratios...).
- Es **fácil de programar** e **intuitivo de explicar**.

#### Analogía

> Imagina que tienes una bolsa con 100 canicas de colores. Quieres saber qué porcentaje son rojas, pero solo puedes tomar un puñado a la vez.  
> Bootstrap = sacar muchos puñados (devolviendo las canicas cada vez), anotar el % de rojas en cada puñado, y ver cuánto varía ese porcentaje.

### Objetivo
Usar bootstrap para estimar el **intervalo de confianza del ticket promedio** para cada modelo de pricing, y la **diferencia entre ambos**.

### Pistas
- `rng.choice(datos, size=len(datos), replace=True)` genera una muestra con reemplazo.
- Guarda cada resultado en una lista y al final convierte a array para analizar.
- Usa `np.percentile()` para obtener los límites del intervalo.


In [ ]:
# ============================================================
# Ejercicio 2a: Bootstrap del ticket promedio para un grupo
# ============================================================
# Paso a paso, primero para el grupo "premium"

# 1) Extraer los tickets del modelo premium (sin NaN)
tickets_premium = df.loc[df["modelo_pricing"] == "premium", "ticket_usd"].dropna().values

print(f"Observaciones premium (sin NaN): {len(tickets_premium)}")
print(f"Ticket promedio observado: ${tickets_premium.mean():.2f}")
print()

# 2) Configurar el bootstrap
n_bootstrap = 5000      # número de remuestreos
rng_boot = np.random.default_rng(RANDOM_SEED)

# 3) Ejecutar el bootstrap
medias_bootstrap = []   # aquí guardaremos cada media calculada

for i in range(n_bootstrap):
    # Muestra con reemplazo del mismo tamaño que los datos originales
    muestra = rng_boot.choice(tickets_premium, size=len(tickets_premium), replace=True)
    medias_bootstrap.append(muestra.mean())

# Convertir a array de numpy
medias_bootstrap = np.array(medias_bootstrap)

print(f"Bootstrap completado: {n_bootstrap} remuestreos")
print(f"Media de las medias bootstrap: ${medias_bootstrap.mean():.2f}")
print(f"Desv. estándar de las medias bootstrap: ${medias_bootstrap.std():.2f}")

In [ ]:
# 4) Calcular intervalo de confianza al 95%
#    Usamos los percentiles 2.5 y 97.5 de la distribución bootstrap

ic_inferior = np.percentile(medias_bootstrap, 2.5)
ic_superior = np.percentile(medias_bootstrap, 97.5)

print(f"Intervalo de confianza al 95% para el ticket premium:")
print(f"  [{ic_inferior:.2f}, {ic_superior:.2f}] USD")
print()
print(f"Interpretación: tenemos 95% de confianza en que el ticket")
print(f"promedio REAL del modelo premium está entre ${ic_inferior:.2f} y ${ic_superior:.2f}.")

In [ ]:
# 5) Visualizar la distribución bootstrap

plt.figure(figsize=(10, 5))

plt.hist(medias_bootstrap, bins=50, color="steelblue", edgecolor="white", alpha=0.7)

# Líneas del intervalo de confianza
plt.axvline(ic_inferior, color="red", linestyle="--", linewidth=2, label=f"IC inferior: ${ic_inferior:.2f}")
plt.axvline(ic_superior, color="red", linestyle="--", linewidth=2, label=f"IC superior: ${ic_superior:.2f}")

# Media observada
plt.axvline(tickets_premium.mean(), color="orange", linewidth=2, label=f"Media observada: ${tickets_premium.mean():.2f}")

plt.xlabel("Ticket promedio (USD)")
plt.ylabel("Frecuencia")
plt.title("Distribución bootstrap del ticket promedio — Modelo Premium")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Ejercicio 2b: Bootstrap de la DIFERENCIA entre premium y estándar
# ============================================================
# Esta es la pregunta de negocio: ¿cuánto más paga un cliente premium?

tickets_estandar = df.loc[df["modelo_pricing"] == "estandar", "ticket_usd"].dropna().values

print(f"Observaciones estándar: {len(tickets_estandar)}")
print(f"Observaciones premium:  {len(tickets_premium)}")
print(f"Diferencia observada: ${tickets_premium.mean() - tickets_estandar.mean():.2f}")
print()

# Bootstrap de la diferencia
n_bootstrap = 5000
rng_boot2 = np.random.default_rng(RANDOM_SEED)

diferencias_bootstrap = []

for i in range(n_bootstrap):
    # Remuestrear CADA grupo por separado
    muestra_premium = rng_boot2.choice(tickets_premium, size=len(tickets_premium), replace=True)
    muestra_estandar = rng_boot2.choice(tickets_estandar, size=len(tickets_estandar), replace=True)
    
    # Diferencia de medias en esta iteración
    diferencia = muestra_premium.mean() - muestra_estandar.mean()
    diferencias_bootstrap.append(diferencia)

diferencias_bootstrap = np.array(diferencias_bootstrap)

# Intervalo de confianza
ic_diff_inf = np.percentile(diferencias_bootstrap, 2.5)
ic_diff_sup = np.percentile(diferencias_bootstrap, 97.5)

print(f"IC 95% para la diferencia (premium - estándar):")
print(f"  [{ic_diff_inf:.2f}, {ic_diff_sup:.2f}] USD")
print()

# ¿El intervalo incluye el cero?
if ic_diff_inf > 0:
    print("El intervalo NO incluye el 0 → hay evidencia de que premium tiene ticket mayor.")
else:
    print("El intervalo INCLUYE el 0 → no podemos asegurar diferencia.")

In [ ]:
# Visualizar la distribución de diferencias

plt.figure(figsize=(10, 5))

plt.hist(diferencias_bootstrap, bins=50, color="coral", edgecolor="white", alpha=0.7)
plt.axvline(0, color="black", linestyle="-", linewidth=2, label="Sin diferencia (0)")
plt.axvline(ic_diff_inf, color="red", linestyle="--", linewidth=2, label=f"IC inf: ${ic_diff_inf:.2f}")
plt.axvline(ic_diff_sup, color="red", linestyle="--", linewidth=2, label=f"IC sup: ${ic_diff_sup:.2f}")

plt.xlabel("Diferencia de ticket promedio (premium − estándar) en USD")
plt.ylabel("Frecuencia")
plt.title("Distribución bootstrap de la diferencia de medias")
plt.legend()
plt.tight_layout()
plt.show()

> **Punto clave:** el bootstrap nos dio un **rango de valores plausibles** para la diferencia, SIN necesitar fórmulas de distribuciones. Si el intervalo no incluye el 0, tenemos evidencia de una diferencia real.

> **Pregunta para la clase:** ¿Qué pasaría si en vez de 8,000 observaciones tuviéramos solo 50?  
> (Pista: el intervalo sería mucho más ancho → más incertidumbre).


---
## Ejercicio 3 · Simulación de Monte Carlo: estimar probabilidades (20 min)

### Introducción teórica

**Monte Carlo** es una familia de métodos que usan muestreo aleatorio repetido para estimar cantidades difíciles de calcular con fórmulas exactas.

#### ¿Cómo funciona?

1. Define un **modelo** del proceso que quieres estudiar (con sus reglas y parámetros).
2. **Simula** el proceso miles de veces, cada vez con valores aleatorios.
3. **Agrega** los resultados para obtener estimaciones (promedio, probabilidad, percentiles).

#### Aplicaciones en negocio

- **Riesgo financiero:** ¿cuál es la probabilidad de que mis ingresos caigan más del 20%?
- **Inventario:** ¿cuántas unidades debo tener en stock para cubrir el 95% de los escenarios de demanda?
- **Marketing:** si lanzo una campaña con tasa de conversión estimada, ¿cuál es el rango probable de ingresos?

#### El nombre
> El método se llama "Monte Carlo" por el famoso casino de Mónaco, porque se basa en la aleatoriedad, igual que los juegos de azar.

### Objetivo
Simular los **ingresos mensuales** de la tienda online bajo diferentes escenarios para responder: *¿cuál es la probabilidad de que los ingresos mensuales superen $90,000?*

### Pistas
- Define los parámetros del modelo (visitas/día, tasa de conversión, ticket promedio).
- En cada simulación, usa distribuciones aleatorias para cada componente.
- Repite miles de veces y cuenta cuántas veces se cumple la condición.


In [ ]:
# ============================================================
# Ejercicio 3a: Simulación Monte Carlo de ingresos mensuales
# ============================================================

# Parámetros del modelo (basados en nuestro dataset)
# Los estimamos a partir de los datos generados

# Visitas diarias: usamos Poisson (conteo de eventos por período)
visitas_media_diaria = 150  # promedio de visitas por día

# Tasa de conversión: la estimamos de los datos
tasa_conv_media = df["convertido"].mean()
print(f"Tasa de conversión estimada: {tasa_conv_media:.4f} ({tasa_conv_media*100:.1f}%)")

# Ticket promedio y su desviación: los estimamos de los datos
ticket_medio = df["ticket_usd"].mean()
ticket_std = df["ticket_usd"].std()
print(f"Ticket medio estimado: ${ticket_medio:.2f}")
print(f"Desv. estándar del ticket: ${ticket_std:.2f}")

In [ ]:
# ============================================================
# Simular 10,000 meses
# ============================================================
# Cada "mes" tiene 30 días. En cada día:
#   1. Llegan X visitas (Poisson)
#   2. Cada visita convierte con probabilidad p (Binomial)
#   3. Cada conversión genera un ticket aleatorio (Normal)
#   4. Ingreso del día = suma de tickets de conversiones
#   5. Ingreso mensual = suma de 30 días

n_simulaciones = 10_000
dias_por_mes = 30
rng_mc = np.random.default_rng(RANDOM_SEED)

ingresos_mensuales = []

for sim in range(n_simulaciones):
    ingreso_mes = 0
    
    for dia in range(dias_por_mes):
        # 1. ¿Cuántas visitas hoy?
        visitas_hoy = rng_mc.poisson(visitas_media_diaria)
        
        # 2. ¿Cuántas convierten?
        conversiones_hoy = rng_mc.binomial(visitas_hoy, tasa_conv_media)
        
        # 3. ¿Cuánto gasta cada uno?
        if conversiones_hoy > 0:
            tickets_hoy = rng_mc.normal(ticket_medio, ticket_std, size=conversiones_hoy)
            tickets_hoy = np.maximum(tickets_hoy, 5)  # mínimo $5
            ingreso_mes += tickets_hoy.sum()
    
    ingresos_mensuales.append(ingreso_mes)

ingresos_mensuales = np.array(ingresos_mensuales)

print(f"Simulación completada: {n_simulaciones} meses simulados")
print(f"Ingreso mensual promedio: ${ingresos_mensuales.mean():,.0f}")
print(f"Ingreso mensual mediana:  ${np.median(ingresos_mensuales):,.0f}")
print(f"Desv. estándar:           ${ingresos_mensuales.std():,.0f}")

In [ ]:
# ============================================================
# Responder preguntas de negocio con la simulación
# ============================================================

meta_ingreso = 90_000

prob_superar_meta = (ingresos_mensuales >= meta_ingreso).mean()

print(f"Pregunta: ¿Cuál es la probabilidad de superar ${meta_ingreso:,} en un mes?")
print(f"Respuesta: {prob_superar_meta:.1%}")
print()

# Percentiles útiles para el negocio
print("Escenarios de ingreso mensual:")
for percentil in [5, 25, 50, 75, 95]:
    valor = np.percentile(ingresos_mensuales, percentil)
    print(f"  Percentil {percentil:3d}: ${valor:>10,.0f}")

print()
print(f"En el peor 5% de los casos, el ingreso sería menor a ${np.percentile(ingresos_mensuales, 5):,.0f}")
print(f"En el mejor 5%, el ingreso superaría ${np.percentile(ingresos_mensuales, 95):,.0f}")

In [ ]:
# Visualización completa

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Histograma ---
axes[0].hist(ingresos_mensuales, bins=60, color="steelblue", edgecolor="white", alpha=0.7)
axes[0].axvline(meta_ingreso, color="red", linestyle="--", linewidth=2, 
                label=f"Meta: ${meta_ingreso:,}")
axes[0].axvline(ingresos_mensuales.mean(), color="orange", linewidth=2, 
                label=f"Media: ${ingresos_mensuales.mean():,.0f}")
axes[0].set_xlabel("Ingreso mensual (USD)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución de ingresos mensuales simulados")
axes[0].legend()

# --- CDF (probabilidad acumulada) ---
ingresos_sorted = np.sort(ingresos_mensuales)
cdf = np.arange(1, len(ingresos_sorted) + 1) / len(ingresos_sorted)

axes[1].plot(ingresos_sorted, cdf, color="steelblue", linewidth=1.5)
axes[1].axvline(meta_ingreso, color="red", linestyle="--", linewidth=2)
axes[1].axhline(1 - prob_superar_meta, color="gray", linestyle=":", alpha=0.5)
axes[1].set_xlabel("Ingreso mensual (USD)")
axes[1].set_ylabel("Probabilidad acumulada")
axes[1].set_title("CDF: ¿cuánta probabilidad hay debajo de cada valor?")
axes[1].annotate(f"P(ingreso ≥ ${meta_ingreso:,}) = {prob_superar_meta:.1%}",
                 xy=(meta_ingreso, 1 - prob_superar_meta),
                 xytext=(meta_ingreso + 3000, 0.3),
                 arrowprops=dict(arrowstyle="->"),
                 fontsize=10, color="red")

plt.tight_layout()
plt.show()

> **Punto clave:** Monte Carlo nos permitió responder una pregunta de negocio compleja ("¿probabilidad de superar $90K?") sin necesitar fórmulas cerradas. Solo necesitamos:
> 1. Un modelo del proceso (visitas → conversiones → tickets).
> 2. Distribuciones razonables para cada componente.
> 3. Muchas repeticiones.

> **Pregunta para la clase:** ¿Qué parámetro cambiarías si quisieras simular el efecto de una nueva campaña de marketing que aumenta las visitas en un 30%?


---
## Ejercicio 4 · Test de permutaciones: hipótesis sin supuestos (20 min)

### Introducción teórica

El **test de permutaciones** es una prueba de hipótesis que no asume ninguna distribución (no necesita normalidad, ni igualdad de varianzas). Es una alternativa poderosa cuando los supuestos de las pruebas clásicas (t-test, z-test) no se cumplen.

#### ¿Cómo funciona?

La idea central es: **si no hubiera diferencia real entre los grupos, entonces las etiquetas (grupo A, grupo B) no importarían.**

1. Calcula el estadístico observado (ej.: diferencia de medias entre premium y estándar).
2. **Mezcla** (permuta) las etiquetas de grupo al azar: asigna aleatoriamente quién es "premium" y quién es "estándar".
3. Calcula el mismo estadístico con las etiquetas mezcladas.
4. Repite 2-3 muchas veces (ej.: 10,000).
5. **p-value** = proporción de veces que el estadístico permutado fue tan extremo o más que el observado.

#### ¿Cuándo usar permutaciones vs. t-test?

| Situación | Mejor opción |
|---|---|
| Muestra grande, datos simétricos | t-test (más rápido, misma conclusión) |
| Muestra pequeña (n < 30) | Permutaciones (no necesita normalidad) |
| Datos con outliers extremos | Permutaciones (más robusto) |
| Estadístico raro (mediana, ratio, etc.) | Permutaciones (t-test solo sirve para medias) |

### Objetivo
Determinar si la diferencia de ticket promedio entre premium y estándar es **estadísticamente significativa**, usando un test de permutaciones.

### Pistas
- `rng.permutation()` mezcla un array al azar.
- La idea clave: si las etiquetas no importaran, la diferencia observada sería fácil de obtener "por azar".
- Si la diferencia observada cae en las colas extremas de la distribución de permutaciones, es evidencia de un efecto real.


In [ ]:
# ============================================================
# Ejercicio 4a: Preparar datos para el test de permutaciones
# ============================================================

# Trabajamos con ticket_usd sin NaN
df_clean = df[["modelo_pricing", "ticket_usd"]].dropna()

grupo_premium = df_clean.loc[df_clean["modelo_pricing"] == "premium", "ticket_usd"].values
grupo_estandar = df_clean.loc[df_clean["modelo_pricing"] == "estandar", "ticket_usd"].values

# Estadístico observado: diferencia de medias
diff_observada = grupo_premium.mean() - grupo_estandar.mean()

print(f"Media premium:  ${grupo_premium.mean():.2f}  (n={len(grupo_premium)})")
print(f"Media estándar: ${grupo_estandar.mean():.2f}  (n={len(grupo_estandar)})")
print(f"Diferencia observada: ${diff_observada:.2f}")

In [ ]:
# ============================================================
# Ejercicio 4b: Ejecutar el test de permutaciones
# ============================================================

# Juntamos todos los valores en un solo array
todos_los_tickets = np.concatenate([grupo_premium, grupo_estandar])
n_premium = len(grupo_premium)
n_total = len(todos_los_tickets)

n_permutaciones = 10_000
rng_perm = np.random.default_rng(RANDOM_SEED)

diferencias_perm = []

for i in range(n_permutaciones):
    # Mezclar las etiquetas: los primeros n_premium son "premium", el resto "estándar"
    indices_mezclados = rng_perm.permutation(n_total)
    tickets_mezclados = todos_los_tickets[indices_mezclados]
    
    # Separar en dos grupos con las nuevas "etiquetas"
    perm_premium = tickets_mezclados[:n_premium]
    perm_estandar = tickets_mezclados[n_premium:]
    
    # Calcular la diferencia de medias con etiquetas mezcladas
    diff_perm = perm_premium.mean() - perm_estandar.mean()
    diferencias_perm.append(diff_perm)

diferencias_perm = np.array(diferencias_perm)

print(f"Test de permutaciones completado: {n_permutaciones} permutaciones")

In [ ]:
# ============================================================
# Ejercicio 4c: Calcular el p-value
# ============================================================

# p-value bilateral: ¿cuántas permutaciones dieron una diferencia
# tan extrema (en valor absoluto) como la observada?
p_value_perm = (np.abs(diferencias_perm) >= np.abs(diff_observada)).mean()

print(f"Diferencia observada: ${diff_observada:.2f}")
print(f"p-value (permutaciones): {p_value_perm:.6f}")
print()

alpha = 0.05
if p_value_perm < alpha:
    print(f"Con α={alpha}: RECHAZAMOS H0.")
    print("Hay evidencia de que la diferencia de ticket entre premium y estándar es real.")
else:
    print(f"Con α={alpha}: NO rechazamos H0.")
    print("No hay evidencia suficiente de una diferencia real.")

In [ ]:
# ============================================================
# Ejercicio 4d: Comparar con el t-test clásico
# ============================================================

t_stat, p_value_ttest = stats.ttest_ind(grupo_premium, grupo_estandar)

print("Comparación de métodos:")
print(f"  Test de permutaciones → p-value: {p_value_perm:.6f}")
print(f"  t-test clásico        → p-value: {p_value_ttest:.6f}")
print()
print("En este caso los p-values son similares porque la muestra es grande")
print("y los datos son aproximadamente normales. La ventaja del test de")
print("permutaciones se nota más con muestras pequeñas o datos no normales.")

In [ ]:
# Visualización del test de permutaciones

plt.figure(figsize=(10, 5))

plt.hist(diferencias_perm, bins=60, color="lightgray", edgecolor="white", 
         alpha=0.8, label="Distribución bajo H0 (permutaciones)")

# Marcar la diferencia observada
plt.axvline(diff_observada, color="red", linewidth=2.5, 
            label=f"Diferencia observada: ${diff_observada:.2f}")
plt.axvline(-diff_observada, color="red", linewidth=2.5, linestyle="--",
            label=f"Espejo: -${diff_observada:.2f}")

plt.xlabel("Diferencia de medias (permutada)")
plt.ylabel("Frecuencia")
plt.title(f"Test de permutaciones — p-value = {p_value_perm:.4f}")
plt.legend()
plt.tight_layout()
plt.show()

print("La línea roja muestra dónde cae nuestra diferencia observada.")
print("Si cae en una zona donde pocas permutaciones llegan, es evidencia de un efecto real.")

> **Punto clave:** el test de permutaciones construye su propia distribución de referencia "bajo H0" al mezclar las etiquetas. No necesita asumir normalidad. Es como preguntarse: *"si premium y estándar fueran lo mismo, ¿sería fácil ver una diferencia así de grande por azar?"*

> **Recurso externo:** para una explicación visual e interactiva del test de permutaciones:  
> [Seeing Theory – Inferencia Estadística (Brown University)](https://seeing-theory.brown.edu/frequentist-inference/index.html)  


---
## Proyecto integrador · ¿El nuevo pricing es mejor? (15 min)

### Contexto

El equipo de producto quiere saber si el modelo de pricing "premium" genera más ingresos **totales** que el modelo "estándar". No solo importa el ticket promedio — también importa la tasa de conversión.

Vamos a usar **TODAS** las técnicas de la clase para dar una respuesta completa:

1. **Datos sintéticos** → ya los tenemos (nuestro dataset).
2. **Bootstrap** → intervalo de confianza para el ingreso por visita.
3. **Monte Carlo** → proyectar ingresos mensuales para cada modelo.
4. **Permutaciones** → validar si la diferencia es significativa.

### Métrica clave: Ingreso por visita (RPV)

El **Revenue Per Visit (RPV)** combina conversión y ticket:

$$\text{RPV} = \text{Tasa de conversión} \times \text{Ticket promedio}$$

Esta es la métrica que realmente le importa al negocio.


In [ ]:
# ============================================================
# Proyecto integrador - Paso 1: Calcular RPV observado
# ============================================================

# Rellenar ticket_usd con 0 para quienes no convirtieron
# (su ingreso para la tienda es $0)
df["ingreso_visita"] = df["ticket_usd"].fillna(0) * df["convertido"]

rpv_premium = df.loc[df["modelo_pricing"] == "premium", "ingreso_visita"].values
rpv_estandar = df.loc[df["modelo_pricing"] == "estandar", "ingreso_visita"].values

print("=== Revenue Per Visit (RPV) observado ===")
print(f"RPV Premium:  ${rpv_premium.mean():.2f}")
print(f"RPV Estándar: ${rpv_estandar.mean():.2f}")
print(f"Diferencia:   ${rpv_premium.mean() - rpv_estandar.mean():.2f}")

In [ ]:
# ============================================================
# Proyecto integrador - Paso 2: Bootstrap del RPV por grupo
# ============================================================

n_boot = 5000
rng_proj = np.random.default_rng(RANDOM_SEED)

boot_rpv_premium = []
boot_rpv_estandar = []
boot_rpv_diff = []

for i in range(n_boot):
    sample_p = rng_proj.choice(rpv_premium, size=len(rpv_premium), replace=True)
    sample_e = rng_proj.choice(rpv_estandar, size=len(rpv_estandar), replace=True)
    
    boot_rpv_premium.append(sample_p.mean())
    boot_rpv_estandar.append(sample_e.mean())
    boot_rpv_diff.append(sample_p.mean() - sample_e.mean())

boot_rpv_premium = np.array(boot_rpv_premium)
boot_rpv_estandar = np.array(boot_rpv_estandar)
boot_rpv_diff = np.array(boot_rpv_diff)

print("=== Bootstrap IC 95% para RPV ===")
print(f"RPV Premium:  [{np.percentile(boot_rpv_premium, 2.5):.2f}, {np.percentile(boot_rpv_premium, 97.5):.2f}]")
print(f"RPV Estándar: [{np.percentile(boot_rpv_estandar, 2.5):.2f}, {np.percentile(boot_rpv_estandar, 97.5):.2f}]")
print(f"Diferencia:   [{np.percentile(boot_rpv_diff, 2.5):.2f}, {np.percentile(boot_rpv_diff, 97.5):.2f}]")
print()

if np.percentile(boot_rpv_diff, 2.5) > 0:
    print("El IC de la diferencia NO incluye 0 → Premium genera más ingreso por visita.")
else:
    print("El IC de la diferencia INCLUYE 0 → no hay evidencia clara de ventaja.")

In [ ]:
# ============================================================
# Proyecto integrador - Paso 3: Monte Carlo comparativo
# ============================================================
# Simulamos ingresos mensuales para cada modelo por separado

n_sim = 5000
dias = 30
visitas_dia = 150
rng_mc2 = np.random.default_rng(RANDOM_SEED)

def simular_ingreso_mensual(rpv_datos, n_sim, dias, visitas_dia, rng):
    """Simula ingresos mensuales a partir del RPV de un grupo."""
    resultados = []
    for _ in range(n_sim):
        ingreso = 0
        for _ in range(dias):
            # Visitas del día
            n_visitas = rng.poisson(visitas_dia)
            # Tomar una muestra del RPV real
            rpv_sample = rng.choice(rpv_datos, size=n_visitas, replace=True)
            ingreso += rpv_sample.sum()
        resultados.append(ingreso)
    return np.array(resultados)

ingresos_premium = simular_ingreso_mensual(rpv_premium, n_sim, dias, visitas_dia, rng_mc2)
ingresos_estandar = simular_ingreso_mensual(rpv_estandar, n_sim, dias, visitas_dia, rng_mc2)

print("=== Ingresos mensuales simulados ===")
print(f"Premium:  media=${ingresos_premium.mean():>10,.0f}  |  mediana=${np.median(ingresos_premium):>10,.0f}")
print(f"Estándar: media=${ingresos_estandar.mean():>10,.0f}  |  mediana=${np.median(ingresos_estandar):>10,.0f}")
print()

prob_premium_gana = (ingresos_premium > ingresos_estandar).mean()
print(f"Probabilidad de que Premium supere a Estándar en un mes dado: {prob_premium_gana:.1%}")

In [ ]:
# ============================================================
# Proyecto integrador - Paso 4: Test de permutaciones sobre RPV
# ============================================================

diff_rpv_obs = rpv_premium.mean() - rpv_estandar.mean()
todos_rpv = np.concatenate([rpv_premium, rpv_estandar])
n_prem = len(rpv_premium)

n_perm = 10_000
rng_perm2 = np.random.default_rng(RANDOM_SEED)

diff_perm_rpv = []
for _ in range(n_perm):
    idx = rng_perm2.permutation(len(todos_rpv))
    rpv_shuffled = todos_rpv[idx]
    d = rpv_shuffled[:n_prem].mean() - rpv_shuffled[n_prem:].mean()
    diff_perm_rpv.append(d)

diff_perm_rpv = np.array(diff_perm_rpv)
p_val_rpv = (np.abs(diff_perm_rpv) >= np.abs(diff_rpv_obs)).mean()

print(f"Diferencia RPV observada: ${diff_rpv_obs:.3f}")
print(f"p-value (permutaciones):  {p_val_rpv:.6f}")
print()

if p_val_rpv < 0.05:
    print("La diferencia de RPV es estadísticamente significativa (p < 0.05).")
else:
    print("La diferencia de RPV NO es estadísticamente significativa (p >= 0.05).")

In [ ]:
# ============================================================
# Proyecto integrador - Paso 5: Resumen visual ejecutivo
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Panel 1: Bootstrap RPV ---
axes[0, 0].hist(boot_rpv_diff, bins=50, color="steelblue", edgecolor="white", alpha=0.7)
axes[0, 0].axvline(0, color="black", linestyle="-", linewidth=2, label="Sin diferencia")
axes[0, 0].axvline(np.percentile(boot_rpv_diff, 2.5), color="red", linestyle="--")
axes[0, 0].axvline(np.percentile(boot_rpv_diff, 97.5), color="red", linestyle="--")
axes[0, 0].set_title("1. Bootstrap: Diferencia RPV (premium - estándar)")
axes[0, 0].set_xlabel("Diferencia RPV (USD)")
axes[0, 0].legend()

# --- Panel 2: Monte Carlo comparativo ---
axes[0, 1].hist(ingresos_premium, bins=50, alpha=0.5, color="green", label="Premium")
axes[0, 1].hist(ingresos_estandar, bins=50, alpha=0.5, color="orange", label="Estándar")
axes[0, 1].set_title("2. Monte Carlo: Distribución de ingresos mensuales")
axes[0, 1].set_xlabel("Ingreso mensual (USD)")
axes[0, 1].legend()

# --- Panel 3: Permutaciones ---
axes[1, 0].hist(diff_perm_rpv, bins=50, color="lightgray", edgecolor="white")
axes[1, 0].axvline(diff_rpv_obs, color="red", linewidth=2.5, label=f"Observado: ${diff_rpv_obs:.3f}")
axes[1, 0].set_title(f"3. Test de permutaciones (p={p_val_rpv:.4f})")
axes[1, 0].set_xlabel("Diferencia RPV (permutada)")
axes[1, 0].legend()

# --- Panel 4: Resumen de decisión ---
axes[1, 1].axis("off")
resumen_texto = (
    f"RESUMEN EJECUTIVO\n"
    f"{'='*35}\n\n"
    f"RPV Premium:  ${rpv_premium.mean():.2f}\n"
    f"RPV Estándar: ${rpv_estandar.mean():.2f}\n"
    f"Diferencia:   ${diff_rpv_obs:.2f}\n\n"
    f"IC 95% diferencia:\n"
    f"  [{np.percentile(boot_rpv_diff, 2.5):.2f}, {np.percentile(boot_rpv_diff, 97.5):.2f}]\n\n"
    f"p-value (permutaciones): {p_val_rpv:.4f}\n\n"
    f"P(Premium > Estándar en\n"
    f"  un mes dado): {prob_premium_gana:.1%}\n\n"
    f"Ingreso mensual esperado:\n"
    f"  Premium:  ${ingresos_premium.mean():,.0f}\n"
    f"  Estándar: ${ingresos_estandar.mean():,.0f}"
)
axes[1, 1].text(0.1, 0.95, resumen_texto, transform=axes[1, 1].transAxes,
                fontsize=12, verticalalignment="top", fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.suptitle("Análisis completo por simulación: ¿Premium es mejor?", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Takeaways (5 min)

1. **Simular = generar datos con reglas conocidas** para explorar, estimar y validar.
2. **Datos sintéticos** nos dejan practicar y prototipar con total control (sabemos "la verdad").
3. **Bootstrap** da intervalos de confianza para CUALQUIER estadístico, sin fórmulas ni supuestos de distribución.
4. **Monte Carlo** permite responder preguntas complejas de negocio (probabilidades, riesgo, escenarios) simulando el proceso completo muchas veces.
5. **Test de permutaciones** es una alternativa poderosa al t-test cuando los datos no cumplen supuestos — construye su propia distribución "bajo H0".
6. En la práctica, estas técnicas se **combinan**: generas datos → bootstrapeas → simulas escenarios → validas con permutaciones.


## Cierre y próximos pasos

Temas que se pueden explorar después de esta sesión:
- **Validación cruzada** (otra forma de remuestreo, usada en machine learning).
- **Simulación de series temporales** (agregar tendencia, estacionalidad).
- **Optimización por simulación** (encontrar el mejor precio, inventario, etc.).
- **Bayesian bootstrap** (variante con pesos aleatorios en vez de remuestreo).

### Recursos adicionales para profundizar

- [GeoGebra – Estadística interactiva](https://www.geogebra.org/math/statistics?lang=en): visualizar distribuciones y muestras.
- [Seeing Theory (Brown University)](https://seeing-theory.brown.edu/): visualizaciones interactivas de conceptos de probabilidad y estadística.
- [Statistics by Jim – Bootstrapping](https://statisticsbyjim.com/hypothesis-testing/bootstrapping/): explicación clara de bootstrap con ejemplos.

### Pregunta final

Si el equipo de finanzas te pide *"¿cuánto podríamos perder si la tasa de conversión cae un 20% el próximo trimestre?"*, ¿qué técnica de simulación usarías y por qué?
